# 5. The Scoring Engine

Part of the `nuclearff` [User Tutorial](https://nolmacdonald.github.io/nuclearff/tutorial/index.html) notebook series -- every notebook shares one running `./demo` directory and depends on the ones before it having already run. See [README.md](README.md) for the full run order. Run `04_capturing_a_league.ipynb` first.

`nflreadpy`'s own `fantasy_points` column (`01_introduction.ipynb`) uses one generic scoring formula. Real leagues don't all score the same way -- this league gives a full point per reception and 6 points per passing touchdown; another league might run half-PPR and 4-point passing touchdowns. `ScoringEngine` closes that gap: it applies *this league's own* `ScoringSettings` (`02_configuration.ipynb`) to real nflverse stat lines. Mirrors the [Scoring Engine](https://nolmacdonald.github.io/nuclearff/tutorial/05_scoring_engine.html) docs page.

## Scoring a single stat line

`score_stat_line` is the reference, hand-verifiable path -- pure Python over one dict of nflverse column names:

In [1]:
from nuclearff.config import load_league_config
from nuclearff.scoring import ScoringEngine

league_cfg = load_league_config("./demo/configs/leagues/1367225133634191360.yaml")
engine = ScoringEngine(league_cfg.scoring)

line = {"receptions": 8, "receiving_yards": 100, "receiving_tds": 1, "position": "WR"}
engine.score_stat_line(line)

ScoringEngine has no mapping for 48 nonzero scoring key(s); these will not be counted toward fantasy points: blk_kick=2.0, blk_kick_ret_yd=0.04, def_3_and_out=0.25, def_4_and_stop=0.5, def_pr_yd=0.04, def_st_ff=1.0, def_st_fum_rec=1.0, def_st_td=6.0, def_td=6.0, ff=1.0, fg_ret_yd=0.04, fgm_0_19=3.0, fgm_20_29=3.0, fgm_30_39=3.0, fgm_40_49=4.0, fgm_50p=5.0, fgmiss=-1.0, fgmiss_0_19=-1.0, fgmiss_20_29=-1.0, fum_rec=2.0, fum_rec_td=6.0, fum_ret_yd=0.04, int=2.0, int_ret_yd=0.04, kr_yd=0.04, pr_yd=0.04, pts_allow_0=5.0, pts_allow_14_20=1.0, pts_allow_1_6=4.0, pts_allow_28_34=-1.0, pts_allow_35p=-4.0, pts_allow_7_13=3.0, sack=1.0, safe=2.0, st_ff=1.0, st_fum_rec=1.0, st_td=6.0, tkl_loss=0.25, xpm=1.0, xpmiss=-1.0, yds_allow_0_100=5.0, yds_allow_100_199=3.0, yds_allow_200_299=2.0, yds_allow_350_399=-1.0, yds_allow_400_449=-3.0, yds_allow_450_499=-5.0, yds_allow_500_549=-6.0, yds_allow_550p=-7.0


24.0

A stat the line doesn't include is treated as zero -- you don't need to pad out every possible key, just the ones a real player recorded. Pass `"position"` to get position-conditional reception bonuses (`bonus_rec_wr`/`bonus_rec_te`/`bonus_rec_rb`) -- without it, those bonuses contribute nothing.

## Scoring a real season, vectorized

`score_frame` is the vectorized equivalent, built for real multi-row data straight out of `nflreadpy`:

In [2]:
from nuclearff.nflverse import load_seasonal_skill_stats

stats = load_seasonal_skill_stats([2024])
scored = engine.score_frame(stats)

scored.select(["player_display_name", "position", "season", "fantasy_points"]).sort(
    "fantasy_points", descending=True
).head(6)

player_display_name,position,season,fantasy_points
str,str,i32,f64
"""Lamar Jackson""","""QB""",2024,512.38
"""Joe Burrow""","""QB""",2024,458.82
"""Baker Mayfield""","""QB""",2024,447.8
"""Josh Allen""","""QB""",2024,435.04
"""Jayden Daniels""","""QB""",2024,405.82
"""Ja'Marr Chase""","""WR""",2024,403.0


That's this league's real 6-point passing touchdowns and full-PPR scoring showing up immediately: quarterbacks dominate the top of the 2024 season under these rules in a way a half-PPR, 4-point-passing-touchdown league would not produce.

> **Why this matters:** it's a common mistake to reach for a league's `scoring` settings dict directly and hand-roll the math yourself for "just one quick check." `ScoringEngine` exists precisely because that math has more edge cases than it looks like -- position-conditional bonuses, unscored keys (below), and keeping every downstream chapter (projections, valuation, backtesting) using the *exact same* scoring logic instead of each reimplementing a slightly different version.

## Unscored keys

A league's `scoring_settings` payload usually includes keys `ScoringEngine` has no nflverse column for -- kicker, defense/special teams, and IDP (individual defensive player) keys, none of which appear in standard offensive box scores. Constructing the engine logs a warning listing every nonzero key it can't account for; `unscored_keys()` returns the same list programmatically:

In [3]:
unscored = engine.unscored_keys()
print(len(unscored), "unscored keys, e.g.:", sorted(unscored)[:5])

48 unscored keys, e.g.: ['blk_kick', 'blk_kick_ret_yd', 'def_3_and_out', 'def_4_and_stop', 'def_pr_yd']


This is expected, not a bug to fix -- recall from `04_capturing_a_league.ipynb` that this exact league's own anomaly detection flagged `no_kicker_or_defense`: it has no kicker, defense, or IDP roster slots at all, so those unscored keys in the payload are inert regardless. A league that *does* roster a kicker or defense would need its own K/DEF stat source joined in before those keys mean anything -- `ScoringEngine` only scores whatever stat columns you hand it.

## What's Next

Raw fantasy points are one number per player-season. `06_metrics.ipynb` computes the underlying opportunity and efficiency signals -- target share, air yards, yards per route run -- that explain *why* a receiver scored what they did, and which of it is likely to repeat.

## See Also

- `02_configuration.ipynb` -- `LeagueConfig`/`ScoringSettings`, the input `ScoringEngine` wraps.
- [Scoring Engine](https://nolmacdonald.github.io/nuclearff/tutorial/05_scoring_engine.html) -- the matching docs page.